# Census data: overview

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

`gerrytools.data` retrieves decennial Census and American Community Survey (ACS) tables,
estimates block-level citizen voting-age population (CVAP), and downloads processed 2020
geographic products. Each retrieval family has its own detailed guide:

- [Decennial PL 94-171](decennial.ipynb) covers the tables behind official redistricting counts.
- [American Community Survey](acs.ipynb) covers ACS estimates, margins of error, and the
  table-definition classes.
- [Block-level CVAP estimates](block_cvap.ipynb) covers the block CVAP estimation method.

This page collects what the three share: API keys, GEOID handling, the processed geographic
downloads, and error handling.

In [ ]:
from gerrytools.data import pl_table

## The Census API key

The [Census Bureau now requires an API key](https://www.census.gov/library/video/2026/adrm/requesting-a-census-data-api-key.html)
for these requests. Every
fetch function accepts `api_key=`; when it is omitted, the key is read from the `CENSUS_API_KEY`
environment variable. Keeping the key in the environment leaves it out of notebooks and shared
configuration.

```python
import os
import us

from gerrytools.data import census

counties = census(
    us.states.GA,
    geometry="county",
    table="P1",
    api_key=os.environ["CENSUS_API_KEY"],
)
```

If neither `api_key=` nor `CENSUS_API_KEY` is set, the call raises a `ValueError` before any
request is made. Because the retrieval guides must build without credentials or network access,
their live calls appear as copyable blocks like the one above, and small committed extracts stand
in for the responses.

> <span class="notebook-admonition-title">Note: Pandas and Census Identifiers </span>
>
> Census geography identifiers are string codes where leading zeroes carry meaning, but pandas has
> a tendency to interpret GEOID columns as integers which drops these leading character. So one
> must take care to read and write GEOIDs as > strings; every `read_csv` call in these guides 
> passes `dtype={"GEOID": "string"}`.
>
> ```python
> example = pd.DataFrame(
>     {
>         "GEOID20": pd.Series(["010010201001", "010010201002"], dtype="string"),
>         "TOTPOP20": [812, 944],
>     }
> )
> example.dtypes
> ```
>
> A GEOID lengthens as the geography gets finer: 2 digits for a state, 5 for a county, 11 for a
> tract, 12 for a block group, and 15 for a block. A block GEOID nests every coarser level, so
> string slicing recovers the parent geography (`block_geoid[:11]` is the tract).

## Column names record their source

Every retrieval function returns GEOID-indexed frames whose column names encode the group, the
measure, and the source vintage. For example, `black_vap_20` comes from decennial P3 and
`black_vap_acs5_23` from the 5-year ACS ending in 2023. Pulls from different tables and vintages 
can therefore coexist in one frame without colliding. The table-definition objects map raw 
Census variables to these names:

In [ ]:
p1_2020 = pl_table("P1", 2020)
list(p1_2020.construct_rename_map(year=2020).items())[:5]

## Rate limits

A rate-limited request (HTTP 429) raises `CensusRateLimitError` rather than a generic HTTP error,
so a retry-later condition is distinguishable from a request that needs correcting; the exception
carries the offending URL and the API's `Retry-After` hint when one is provided.

## Processed 2020 geographic products

`vtds20`, `dualgraphs20`, and `geometries20` download Lab-processed files to a path you choose. 
These files were created prior to 2023, and these interfaces are mainly included for the sake of
preserving old workflows:

```python
from pathlib import Path

import us

from gerrytools.data import dualgraphs20, geometries20, vtds20

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

vtds20(us.states.AL, data_dir / "al_vtds20.zip")
dualgraphs20(us.states.AL, data_dir / "al_dualgraph20.json")
geometries20(us.states.AL, data_dir / "al_blocks20.zip", geometry="block")
```

- `vtds20` retrieves the state's 2020 voting tabulation districts as a zipped shapefile.
- `dualgraphs20` retrieves a ready-made adjacency (dual) graph as GerryChain-loadable JSON.
- `geometries20` retrieves Census geometries; `geometry=` selects the unit level.

The [geometry guide](../geometry.ipynb) covers what to do with these files once they are on disk.

## Related

- [Decennial PL](decennial.ipynb), [ACS](acs.ipynb), and
  [block CVAP](block_cvap.ipynb) walk each retrieval family.
- The [geometry guide](../geometry.ipynb) transforms the retrieved tables and shapes.
- [Data API](../../api/data.rst)